# Dual-Input Diabetic Retinopathy Detection (OCT + OCTA)

This notebook is a full, code-first walkthrough of the pipeline in `fyp/train_dual_input_dr.py`.

It includes:
- complete documentation for each pipeline section
- data integrity checks and leakage-safe splitting
- model training/evaluation logic
- **inline** visualization and reporting graphs
- export to VS Code `# %%` notebook-style `.py` script

> **How to run:** Click **Run All** (or press `Ctrl+Shift+P` → *Run All Cells*).
> Set `DATA_ROOT` in Section 1 to point at your `Final_Dataset` folder.
> If you have no GPU/data yet, set `DEMO_MODE = True` to see example results immediately.


## 1) Project Configuration, Reproducibility, and Runtime Checks

We set seeds, deterministic behaviour, runtime device, output folders,
and version logging so results are traceable.


In [ ]:
import os, sys, random, warnings, json, csv, math
from pathlib import Path
import numpy as np

# ── USER SETTINGS ─────────────────────────────────────────────────────────
# Set to the root of your Final_Dataset folder.
# Expected layout:
#   Final_Dataset/
#       healthy/OCT/*.png   healthy/OCTA/*.png
#       dr/OCT/*.png        dr/OCTA/*.png
DATA_ROOT   = "Final_Dataset"   # ← change this if your folder has a different name
SEED        = 42
OUTPUT_DIR  = Path('outputs')

# Set DEMO_MODE = True to run with SYNTHETIC data and see example results
# immediately (no GPU or real images required).
DEMO_MODE = False

# ── Output folders ────────────────────────────────────────────────────────
for sub in ('figures', 'tables', 'reports', 'checkpoints'):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)

# ── fyp/ modules on path ─────────────────────────────────────────────────
sys.path.insert(0, str(Path("fyp").resolve()))

print('Output directories ready:', [str(OUTPUT_DIR / s) for s in ('figures','tables','reports')])
print(f'DATA_ROOT = {DATA_ROOT!r}  |  DEMO_MODE = {DEMO_MODE}  |  SEED = {SEED}')


## 2) Library Imports and Global Hyperparameter Setup

All core libraries and training constants live in one place for reproducibility.


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# PyTorch (optional — required for training)
try:
    import torch, torchvision
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, WeightedRandomSampler
    import torchvision.models as tv_models
    import torchvision.transforms as transforms
    from PIL import Image
    TORCH_AVAILABLE = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except ImportError:
    TORCH_AVAILABLE = False
    device = None
    print('WARNING: PyTorch not installed.  Training cells will be skipped.')
    print('Install: pip install torch torchvision Pillow')

# scikit-learn (optional — used for ROC/PR AUC)
try:
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix as sk_cm
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print('WARNING: scikit-learn not installed. pip install scikit-learn')

# ── Hyperparameters ───────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 16
LR         = 1e-3
EPOCHS     = 30
PATIENCE   = 7
UNFREEZE_EPOCH = 5
VAL_RATIO  = 0.15
TEST_RATIO = 0.15
MIN_RECALL = 0.90   # minimum recall for screening threshold
AUG_MODE   = 'basic_all'
LOSS_FN    = 'bce'  # 'bce' or 'focal'
CV_FOLDS   = 5
NUM_WORKERS = 0     # set to 4+ on Linux/Mac for speed; keep 0 on Windows

IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__ if TORCH_AVAILABLE else "not available"}')
print(f'\nHyperparameters:')
print(f'  BATCH_SIZE={BATCH_SIZE}, LR={LR}, EPOCHS={EPOCHS}, PATIENCE={PATIENCE}')
print(f'  AUG_MODE={AUG_MODE!r}, LOSS_FN={LOSS_FN!r}, MIN_RECALL={MIN_RECALL}')


## 3) Dataset Structure Check

Verifies the expected folder layout exists before loading.
Expected structure:
```
Final_Dataset/
    healthy/
        OCT/   <patient_id>.png
        OCTA/  <patient_id>.png
    dr/
        OCT/   <patient_id>.png
        OCTA/  <patient_id>.png
```


In [ ]:
root = Path(DATA_ROOT)
issues = []
summary = []

for cls, lbl in [('healthy', 0), ('dr', 1)]:
    cdir = root / cls
    oct_dir  = cdir / 'OCT'
    octa_dir = cdir / 'OCTA'
    if not cdir.exists():
        issues.append(f'Missing class folder: {cdir}')
        continue
    for d in [oct_dir, octa_dir]:
        if not d.exists():
            issues.append(f'Missing subfolder: {d}')
        else:
            files = list(d.glob('*'))
            summary.append(f'  {d}  →  {len(files)} files')

if issues and not DEMO_MODE:
    print('⚠️  Dataset structure issues:')
    for i in issues: print(f'   {i}')
    print('\nFix: set DATA_ROOT to the correct folder, or set DEMO_MODE = True')
elif not issues:
    print(f'✅  Dataset structure OK: {root.resolve()}')
    for s in summary: print(s)
else:
    print('ℹ️  DEMO_MODE=True — using synthetic data (no real images needed)')


## 4) Load Samples and Patient-Level Stratified Split

Pairs each OCT image with its co-registered OCTA by matching filename stems.
Splits are done at the **patient level** so no patient appears in >1 split.


In [ ]:
from collections import defaultdict, namedtuple

Sample = namedtuple('Sample', ['oct_path', 'octa_path', 'label', 'patient_id'])

def build_paired_samples(data_root):
    root = Path(data_root)
    label_dirs = []
    for cls, lbl in [('healthy', 0), ('dr', 1), ('diabetic_retinopathy', 1), ('DR', 1)]:
        cdir = root / cls
        if cdir.exists():
            label_dirs.append((cdir, lbl))
    samples = []
    for class_dir, label in label_dirs:
        oct_dir = class_dir / 'OCT'
        octa_dir = class_dir / 'OCTA'
        if not oct_dir.exists() or not octa_dir.exists():
            raise FileNotFoundError(f'{oct_dir} or {octa_dir} missing')
        oct_files = {p.stem: p for p in oct_dir.glob('*')
                     if p.suffix.lower() in ('.png','.jpg','.tif','.bmp')}
        for stem, oct_path in sorted(oct_files.items()):
            octa_path = None
            for ext in ('.png','.jpg','.tif','.bmp'):
                c = octa_dir / (stem + ext)
                if c.exists():
                    octa_path = c; break
            if octa_path:
                samples.append(Sample(oct_path, octa_path, label, stem))
    return samples

def stratified_split(samples, val_ratio=0.15, test_ratio=0.15, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for s in samples:
        by_label[s.label].append(s.patient_id)
    train_ids, val_ids, test_ids = set(), set(), set()
    for label, pids in by_label.items():
        unique = list(dict.fromkeys(pids))
        rng.shuffle(unique)
        n = len(unique)
        n_test  = max(1, round(n * test_ratio))
        n_val   = max(1, round(n * val_ratio))
        test_ids.update(unique[:n_test])
        val_ids.update(unique[n_test:n_test+n_val])
        train_ids.update(unique[n_test+n_val:])
    return ([s for s in samples if s.patient_id in train_ids],
            [s for s in samples if s.patient_id in val_ids],
            [s for s in samples if s.patient_id in test_ids])

# ── Load (or synthesise) ──────────────────────────────────────────────────
if DEMO_MODE:
    # Synthetic sample list — mimics real dataset distribution
    np.random.seed(SEED)
    _dummy_oct  = Path(OUTPUT_DIR / 'figures' / 'roc_curve_demo.png')
    _dummy_octa = _dummy_oct
    all_samples  = ([Sample(_dummy_oct, _dummy_octa, 0, f'H{i:03d}') for i in range(91)] +
                    [Sample(_dummy_oct, _dummy_octa, 1, f'D{i:03d}') for i in range(35)])
    print('ℹ️  DEMO_MODE: 91 healthy + 35 DR synthetic samples')
else:
    all_samples = build_paired_samples(DATA_ROOT)

train_samples, val_samples, test_samples = stratified_split(
    all_samples, VAL_RATIO, TEST_RATIO, SEED)

print(f'\nTotal samples : {len(all_samples)}  '
      f'(healthy={sum(1 for s in all_samples if s.label==0)}, '
      f'DR={sum(1 for s in all_samples if s.label==1)})')
print(f'Train / Val / Test : {len(train_samples)} / {len(val_samples)} / {len(test_samples)}')


## 5) Class Imbalance Report

Strategy: `WeightedRandomSampler` + `BCEWithLogitsLoss(pos_weight)` to handle the ~2.6:1 healthy-to-DR imbalance.


In [ ]:
def imbalance_report(all_s, train_s, val_s, test_s):
    def _c(ss):
        h = sum(1 for s in ss if s.label==0)
        d = sum(1 for s in ss if s.label==1)
        return h, d, h/max(d,1)
    print('='*55)
    print('CLASS IMBALANCE REPORT')
    print('='*55)
    print(f"{'Split':<12} {'Healthy':>8} {'DR':>8} {'Ratio H:D':>12}")
    print('-'*55)
    for name, ss in [('All',all_s),('Train',train_s),('Val',val_s),('Test',test_s)]:
        h,d,r = _c(ss)
        print(f"{name:<12} {h:>8} {d:>8} {r:>12.2f}")
    print('-'*55)
    print('Strategy: WeightedRandomSampler + BCEWithLogitsLoss(pos_weight)')
    print('='*55)

imbalance_report(all_samples, train_samples, val_samples, test_samples)

# Bar chart — class distribution per split
splits = ['All', 'Train', 'Val', 'Test']
data   = [(sum(1 for s in ss if s.label==0), sum(1 for s in ss if s.label==1))
          for ss in [all_samples, train_samples, val_samples, test_samples]]
x = np.arange(len(splits))
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(x-0.2, [d[0] for d in data], 0.4, label='Healthy', color='steelblue')
ax.bar(x+0.2, [d[1] for d in data], 0.4, label='DR',      color='tomato')
ax.set_xticks(x); ax.set_xticklabels(splits)
ax.set_ylabel('Number of samples'); ax.set_title('Class Distribution per Split')
ax.legend(); plt.tight_layout(); plt.show()


## 6) Sample Image Preview (OCT + OCTA pairs)

Shows one healthy and one DR pair to verify images loaded correctly.


In [ ]:
if not DEMO_MODE and TORCH_AVAILABLE:
    healthy_ex = next((s for s in all_samples if s.label==0), None)
    dr_ex      = next((s for s in all_samples if s.label==1), None)
    fig, axes = plt.subplots(2, 2, figsize=(8, 7))
    for row, (sample, title) in enumerate([(healthy_ex,'Healthy'), (dr_ex,'DR')]):
        if sample:
            axes[row,0].imshow(Image.open(sample.oct_path).convert('RGB'))
            axes[row,0].set_title(f'{title} — OCT'); axes[row,0].axis('off')
            axes[row,1].imshow(Image.open(sample.octa_path).convert('RGB'))
            axes[row,1].set_title(f'{title} — OCTA'); axes[row,1].axis('off')
    plt.suptitle('Sample OCT / OCTA Pairs', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('ℹ️  Image preview skipped (DEMO_MODE or PyTorch/Pillow not available).')
    print('   Run on your real data to see OCT/OCTA images here.')


## 7) Augmentation Policy

Key design rules:
- OCT and OCTA receive the **same** geometric transforms (flip, rotate, translate, scale).
- Intensity transforms (brightness/contrast) are applied to **OCT only** — OCTA vessel intensity encodes physiological information and must not be altered.
- Conservative parameters: rotation ≤ 8°, scale ± 5%, translation ≤ 5 %.


In [ ]:
try:
    from augmentation_policies import PairedAugmentPolicy, AugmentationMode, get_augmentation_policy
    AUG_AVAILABLE = True
    print('✅ augmentation_policies loaded')
    train_policy = get_augmentation_policy(AUG_MODE)
    print(f'   Training policy: {train_policy}')
except ImportError:
    AUG_AVAILABLE = False
    train_policy  = None
    print('⚠️  augmentation_policies.py not on path — using built-in augmentation')

print(f'\nAugmentation mode : {AUG_MODE}')
print('Parameters (BASIC_ALL):')
print('  hflip_p           = 0.50')
print('  rotation_degrees  = ±8°')
print('  brightness_limit  = ±15%  (OCT only)')
print('  contrast_limit    = ±15%  (OCT only)')
print('  translate_frac    = ±5%')
print('  scale_range       = 0.95 – 1.05')


## 8) PyTorch Dataset and DataLoaders

`OCTOCTADatasetV2` loads paired images, applies the augmentation policy, and returns normalised tensors.
`WeightedRandomSampler` oversamples DR cases during training.


In [ ]:
if TORCH_AVAILABLE:
    _BASE_TF = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ])

    class OCTOCTADatasetV2(torch.utils.data.Dataset):
        def __init__(self, samples, policy=None):
            self.samples = samples
            self.policy  = policy
        def __len__(self): return len(self.samples)
        def __getitem__(self, idx):
            s = self.samples[idx]
            if DEMO_MODE:
                # Return random noise tensors in DEMO_MODE
                return (torch.randn(3, IMG_SIZE, IMG_SIZE),
                        torch.randn(3, IMG_SIZE, IMG_SIZE), s.label)
            oct_img  = Image.open(s.oct_path).convert('RGB')
            octa_img = Image.open(s.octa_path).convert('RGB')
            if self.policy is not None:
                oct_img, octa_img = self.policy(oct_img, octa_img, label=s.label)
            return _BASE_TF(oct_img), _BASE_TF(octa_img), s.label

    train_ds = OCTOCTADatasetV2(train_samples, policy=train_policy)
    val_ds   = OCTOCTADatasetV2(val_samples,   policy=None)
    test_ds  = OCTOCTADatasetV2(test_samples,  policy=None)

    cc = [sum(1 for s in train_samples if s.label==c) for c in [0,1]]
    sw = [1.0 / max(cc[s.label],1) for s in train_samples]
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True)

    train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False,   num_workers=NUM_WORKERS)

    print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}')
    print(f'pos_weight (healthy/DR): {cc[0]/max(cc[1],1):.2f}')
else:
    print('PyTorch not available — DataLoader cells will be skipped.')


## 9) Model Architecture: DualResNet

Two independent ResNet-18 backbones process OCT and OCTA separately.
Feature vectors (512-d each) are concatenated → 1024-d → classifier head (256 → 1).

**Phase-1**: only the classifier head is trained (backbones frozen).
**Phase-2**: all parameters fine-tuned at 10× lower LR.


In [ ]:
if TORCH_AVAILABLE:
    class DualResNet(nn.Module):
        def __init__(self, freeze_backbone=True):
            super().__init__()
            b_oct  = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
            b_octa = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
            feat_dim = b_oct.fc.in_features  # 512
            b_oct.fc  = nn.Identity()
            b_octa.fc = nn.Identity()
            self.backbone_oct  = b_oct
            self.backbone_octa = b_octa
            self.classifier = nn.Sequential(
                nn.Linear(feat_dim*2, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, 1))
            if freeze_backbone:
                for p in list(self.backbone_oct.parameters()) + list(self.backbone_octa.parameters()):
                    p.requires_grad = False
        def unfreeze_backbones(self):
            for p in list(self.backbone_oct.parameters()) + list(self.backbone_octa.parameters()):
                p.requires_grad = True
        def forward(self, oct_x, octa_x):
            return self.classifier(torch.cat([self.backbone_oct(oct_x),
                                              self.backbone_octa(octa_x)], dim=1)).squeeze(1)

    model = DualResNet(freeze_backbone=True).to(device)

    # Parameter count
    total   = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters    : {total:,}')
    print(f'Trainable (phase-1) : {trainable:,}  (classifier head only)')
    print(f'Frozen (phase-1)    : {total-trainable:,}  (both ResNet-18 backbones)')
    print(f'\nPhase-2 (epoch {UNFREEZE_EPOCH}+): all {total:,} parameters trained at LR={LR*0.1}')


## 10) Loss Function and Optimizer

**BCEWithLogitsLoss** with `pos_weight = n_healthy / n_DR` penalises missed DR cases more.

**Alternative — Focal Loss:**  
`FL(pₜ) = −αₜ · (1−pₜ)^γ · log(pₜ)`,  γ=2  
Down-weights easy examples and focuses learning on hard misclassifications — beneficial when DR samples are scarce.


In [ ]:
if TORCH_AVAILABLE:
    class BinaryFocalLoss(nn.Module):
        def __init__(self, gamma=2.0, alpha=1.0):
            super().__init__(); self.gamma=gamma; self.alpha=alpha
        def forward(self, logits, targets):
            probs = torch.sigmoid(logits)
            p_t = probs*targets + (1-probs)*(1-targets)
            a_t = self.alpha*targets + (1-targets)
            return (-a_t * (1-p_t)**self.gamma * torch.log(p_t+1e-8)).mean()

    pos_weight_val = cc[0] / max(cc[1], 1)
    pw_tensor = torch.tensor([pos_weight_val], device=device)

    if LOSS_FN == 'focal':
        criterion = BinaryFocalLoss(gamma=2.0, alpha=pos_weight_val)
        print(f'Using BinaryFocalLoss  (gamma=2.0, alpha={pos_weight_val:.3f})')
    else:
        criterion = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
        print(f'Using BCEWithLogitsLoss(pos_weight={pos_weight_val:.3f})')

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=False)
    print(f'Optimizer: Adam  LR={LR}')
    print(f'Scheduler: ReduceLROnPlateau (factor=0.5, patience=3)')


## 11) Training Loop

**Phase 1 (epochs 1–{UNFREEZE_EPOCH-1})**: Only the classifier head is updated.  
**Phase 2 (epoch {UNFREEZE_EPOCH}+)**: Backbone weights are unfrozen and fine-tuned at 10× lower LR.  
Best model is selected by **Balanced Accuracy** on the validation set.
Early stopping halts training after `PATIENCE` epochs without improvement.


In [ ]:
if TORCH_AVAILABLE:
    def run_epoch(model, loader, criterion, optimizer, device, train=True):
        model.train(train)
        total_loss, all_labels, all_probs = 0.0, [], []
        with torch.set_grad_enabled(train):
            for oct_x, octa_x, labels in loader:
                oct_x, octa_x = oct_x.to(device), octa_x.to(device)
                lf = labels.float().to(device)
                logits = model(oct_x, octa_x)
                loss = criterion(logits, lf)
                if train and optimizer:
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                total_loss += loss.item() * len(labels)
                all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy().tolist())
                all_labels.extend(labels.tolist())
        return total_loss/max(len(all_labels),1), all_labels, all_probs

    def binary_metrics(labels, probs, thr=0.5):
        preds = (np.array(probs)>=thr).astype(int); labels=np.array(labels)
        tp=int(((preds==1)&(labels==1)).sum()); tn=int(((preds==0)&(labels==0)).sum())
        fp=int(((preds==1)&(labels==0)).sum()); fn=int(((preds==0)&(labels==1)).sum())
        rec=tp/max(tp+fn,1); spec=tn/max(tn+fp,1)
        return {'recall':rec,'specificity':spec,'precision':tp/max(tp+fp,1),
                'f1':2*tp/max(2*tp+fp+fn,1),'balanced_accuracy':(rec+spec)/2,
                'tp':tp,'tn':tn,'fp':fp,'fn':fn}

    # ── Main training loop ────────────────────────────────────────────────
    best_val_bacc = -1.0
    best_val_probs, best_val_labels, best_epoch = [], [], 0
    patience_count = 0
    train_losses, val_metrics_history = [], []
    best_state = None

    for epoch in range(1, EPOCHS+1):
        if epoch == UNFREEZE_EPOCH:
            model.unfreeze_backbones()
            optimizer = optim.Adam(model.parameters(), lr=LR*0.1)
            print(f'Epoch {epoch:02d} | Backbones unfrozen (phase-2 fine-tuning, LR={LR*0.1})')
        tr_loss, _, _ = run_epoch(model, train_loader, criterion, optimizer, device, True)
        _, v_lbl, v_prb = run_epoch(model, val_loader, criterion, None, device, False)
        vm = binary_metrics(v_lbl, v_prb)
        if SKLEARN_AVAILABLE and len(set(v_lbl))>1:
            vm['roc_auc'] = float(roc_auc_score(v_lbl, v_prb))
        vm['val_loss'] = tr_loss
        train_losses.append(tr_loss)
        val_metrics_history.append(vm)
        print(f"Epoch {epoch:02d} | loss={tr_loss:.4f} | F1={vm['f1']:.3f} "
              f"| Recall={vm['recall']:.3f} | Spec={vm['specificity']:.3f} "
              f"| BAcc={vm['balanced_accuracy']:.3f}")
        scheduler.step(vm['balanced_accuracy'])
        if vm['balanced_accuracy'] > best_val_bacc:
            best_val_bacc = vm['balanced_accuracy']
            best_val_probs, best_val_labels = list(v_prb), list(v_lbl)
            best_epoch = epoch
            patience_count = 0
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (best epoch={best_epoch})')
            break

    print(f'\nBest val balanced-accuracy: {best_val_bacc:.4f} @ epoch {best_epoch}')
else:
    print('PyTorch not available — training skipped.  Using demo metrics below.')


## 12) Training Curves

Loss curve and validation metric curves per epoch.


In [ ]:
if TORCH_AVAILABLE and train_losses:
    epochs_x = list(range(1, len(train_losses)+1))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))

    ax1.plot(epochs_x, train_losses, color='steelblue', label='Train loss')
    val_loss_hist = [m.get('val_loss', float('nan')) for m in val_metrics_history]
    ax1.plot(epochs_x, val_loss_hist, color='orange', linestyle='--', label='Val loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss Curve')
    ax1.legend()

    for key, col in [('recall','tomato'),('specificity','teal'),
                     ('f1','purple'),('balanced_accuracy','darkorange')]:
        vals = [m.get(key, float('nan')) for m in val_metrics_history]
        ax2.plot(epochs_x, vals, label=key, color=col)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score'); ax2.set_title('Validation Metrics')
    ax2.set_ylim([0,1]); ax2.legend()

    plt.suptitle('Training History', fontweight='bold')
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'figures'/'training_curves.png', dpi=150)
    plt.show()
    print('Saved → outputs/figures/training_curves.png')
else:
    # Demo curves
    np.random.seed(42)
    ep = list(range(1,21))
    tr_l = [0.65*np.exp(-0.12*e)+0.05+np.random.uniform(-0.01,0.01) for e in ep]
    recall_h = [min(0.95, 0.55+0.025*e+np.random.uniform(-0.02,0.02)) for e in ep]
    spec_h   = [min(0.97, 0.80+0.008*e+np.random.uniform(-0.01,0.01)) for e in ep]
    f1_h     = [(2*r*p/(r+p+1e-9)) for r,p in zip(recall_h, spec_h)]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
    ax1.plot(ep,tr_l,color='steelblue',label='Train loss')
    ax1.set_xlabel('Epoch');ax1.set_ylabel('Loss');ax1.set_title('Loss Curve (demo)');ax1.legend()
    ax2.plot(ep,recall_h,label='recall',color='tomato')
    ax2.plot(ep,spec_h,  label='specificity',color='teal')
    ax2.plot(ep,f1_h,    label='f1',color='purple')
    ax2.set_xlabel('Epoch');ax2.set_ylabel('Score');ax2.set_title('Val Metrics (demo)')
    ax2.set_ylim([0,1]);ax2.legend()
    plt.suptitle('Training History (demo)',fontweight='bold')
    plt.tight_layout();plt.show()


## 13) Threshold Tuning on Validation Set

The decision threshold is tuned on the **validation set only**.
Objective: maximise **specificity** subject to **recall ≥ MIN_RECALL** (screening safety constraint).


In [ ]:
def find_screening_threshold(labels, probs, min_recall=0.90):
    labels_a, probs_a = np.array(labels), np.array(probs)
    best_t, best_spec = 0.5, -1.0
    fallback_t, fallback_score = 0.5, -1.0
    constraint_met = False
    for t in np.arange(0.01, 1.00, 0.01):
        m = binary_metrics(labels_a, probs_a, t)
        if m['recall'] >= min_recall:
            if not constraint_met or m['specificity'] > best_spec:
                best_t, best_spec, constraint_met = t, m['specificity'], True
        fb = m['balanced_accuracy'] + m['f1']
        if fb > fallback_score:
            fallback_t, fallback_score = t, fb
    if not constraint_met:
        print(f'⚠️  Cannot achieve recall≥{min_recall:.2f}. Falling back to balanced_acc+F1 optimisation.')
        best_t = fallback_t
    return best_t

if TORCH_AVAILABLE and best_val_probs:
    best_threshold = find_screening_threshold(best_val_labels, best_val_probs, MIN_RECALL)
    print(f'Best threshold (from val, min_recall={MIN_RECALL}): {best_threshold:.3f}')
else:
    # DEMO: show threshold sweep chart
    np.random.seed(42)
    N_h, N_d = 91, 35
    demo_probs  = np.concatenate([np.random.beta(2,6,N_h), np.random.beta(5,2,N_d)])
    demo_labels = np.array([0]*N_h+[1]*N_d)
    best_threshold = find_screening_threshold(demo_labels, demo_probs, MIN_RECALL)
    print(f'[DEMO] Best threshold: {best_threshold:.3f}')

# Threshold sweep plot
_labels = demo_labels if (not TORCH_AVAILABLE or not best_val_probs) else np.array(best_val_labels)
_probs  = demo_probs  if (not TORCH_AVAILABLE or not best_val_probs) else np.array(best_val_probs)
ts = np.arange(0.01,1.00,0.01)
recalls=[binary_metrics(_labels,_probs,t)['recall']      for t in ts]
specs  =[binary_metrics(_labels,_probs,t)['specificity'] for t in ts]
f1s    =[binary_metrics(_labels,_probs,t)['f1']          for t in ts]
fig,ax = plt.subplots(figsize=(8,4))
ax.plot(ts,recalls,label='Recall',color='tomato')
ax.plot(ts,specs,  label='Specificity',color='teal')
ax.plot(ts,f1s,    label='F1',color='purple')
ax.axvline(best_threshold, color='black', linestyle='--', label=f'Selected thr={best_threshold:.2f}')
ax.axhline(MIN_RECALL, color='red', linestyle=':', alpha=0.7, label=f'Min recall={MIN_RECALL}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Sweep (Val Set)'); ax.legend()
plt.tight_layout(); plt.show()


## 14) Final Test Set Evaluation

The threshold found on the validation set is applied **unchanged** to the held-out test set.
This prevents threshold-related data leakage.


In [ ]:
if TORCH_AVAILABLE and best_state:
    model.load_state_dict(best_state)
    _, test_labels, test_probs = run_epoch(model, test_loader, criterion, None, device, False)
else:
    # DEMO: simulate realistic test results
    np.random.seed(42)
    N_h_test, N_d_test = 14, 6    # ~15% of 91 healthy, ~15% of 35 DR
    test_probs  = list(np.concatenate([np.random.beta(2,6,N_h_test), np.random.beta(5,2,N_d_test)]))
    test_labels = [0]*N_h_test + [1]*N_d_test
    print('ℹ️  [DEMO] Using synthetic test probabilities')

test_labels_a = np.array(test_labels)
test_probs_a  = np.array(test_probs)
test_preds_a  = (test_probs_a >= best_threshold).astype(int)

test_m = binary_metrics(test_labels, test_probs, best_threshold)
if SKLEARN_AVAILABLE and len(set(test_labels))>1:
    test_m['roc_auc'] = float(roc_auc_score(test_labels, test_probs))
    test_m['pr_auc']  = float(average_precision_score(test_labels, test_probs))
test_m['threshold'] = float(best_threshold)

print('\n' + '='*55)
print(' FINAL TEST SET RESULTS')
print('='*55)
print(f"  ROC AUC          : {test_m.get('roc_auc', 'n/a'):.4f}" if 'roc_auc' in test_m else '  ROC AUC: scikit-learn required')
print(f"  PR  AUC          : {test_m.get('pr_auc', 'n/a'):.4f}" if 'pr_auc' in test_m else '  PR  AUC: scikit-learn required')
print(f"  Threshold        : {test_m['threshold']:.3f}")
print(f"  Recall (Sens.)   : {test_m['recall']:.4f}")
print(f"  Specificity      : {test_m['specificity']:.4f}")
print(f"  Precision        : {test_m['precision']:.4f}")
print(f"  F1 Score         : {test_m['f1']:.4f}")
print(f"  Balanced Acc     : {test_m['balanced_accuracy']:.4f}")
print(f"  TP={test_m['tp']}  TN={test_m['tn']}  FP={test_m['fp']}  FN={test_m['fn']}")
print('='*55)


## 15) ROC Curve

AUC is computed from **continuous probability scores** (not binary predictions) for an unbiased estimate.
The red dot marks the selected operating threshold.


In [ ]:
if SKLEARN_AVAILABLE and len(set(test_labels))>1:
    fpr, tpr, _ = roc_curve(test_labels_a, test_probs_a)
    auc_val = roc_auc_score(test_labels_a, test_probs_a)
    op_m = binary_metrics(test_labels_a, test_probs_a, best_threshold)
    op_fpr, op_tpr = 1-op_m['specificity'], op_m['recall']

    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(fpr, tpr, lw=2, color='steelblue', label=f'ROC (AUC={auc_val:.3f})')
    ax.plot([0,1],[0,1],'k--',lw=1,label='No skill')
    ax.scatter([op_fpr],[op_tpr],color='red',s=80,zorder=5,
               label=f'Operating point (thr={best_threshold:.2f})')
    ax.fill_between(fpr,tpr,alpha=0.08,color='steelblue')
    ax.set_xlabel('False Positive Rate (1 – Specificity)')
    ax.set_ylabel('True Positive Rate (Recall)')
    ax.set_title('Test ROC Curve'); ax.legend(loc='lower right')
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    ax.annotate('Preliminary — small dataset',xy=(0.02,0.02),xycoords='axes fraction',
                fontsize=7,color='grey')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'figures'/'roc_curve.png',dpi=150); plt.show()
    print(f'Saved → outputs/figures/roc_curve.png')
else:
    print('scikit-learn required for ROC curve (pip install scikit-learn)')


## 16) Precision-Recall Curve

PR-AUC is more informative than ROC-AUC for imbalanced datasets.
The dashed baseline shows the no-skill model (prevalence).


In [ ]:
if SKLEARN_AVAILABLE and len(set(test_labels))>1:
    prec, rec, _ = precision_recall_curve(test_labels_a, test_probs_a)
    pr_auc_val = average_precision_score(test_labels_a, test_probs_a)
    prevalence = test_labels_a.mean()
    op_m = binary_metrics(test_labels_a, test_probs_a, best_threshold)

    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(rec, prec, lw=2, color='darkorange', label=f'PR curve (AUC={pr_auc_val:.3f})')
    ax.axhline(prevalence,color='k',linestyle='--',lw=1,
               label=f'No skill (prevalence={prevalence:.2f})')
    ax.scatter([op_m['recall']],[op_m['precision']],color='red',s=80,zorder=5,
               label=f'Operating point (thr={best_threshold:.2f})')
    ax.set_xlabel('Recall (Sensitivity)'); ax.set_ylabel('Precision')
    ax.set_title('Test Precision-Recall Curve'); ax.legend()
    ax.set_xlim([0,1]); ax.set_ylim([0,1])
    ax.annotate('Preliminary — small dataset',xy=(0.02,0.02),xycoords='axes fraction',
                fontsize=7,color='grey')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'figures'/'pr_curve.png',dpi=150); plt.show()
    print('Saved → outputs/figures/pr_curve.png')
else:
    print('scikit-learn required (pip install scikit-learn)')


## 17) Confusion Matrix

- **TP**: DR correctly flagged (critical — missed DR = FN = harm)
- **FN**: DR missed (false negative — most dangerous)
- **FP**: Healthy falsely flagged (causes unnecessary referrals)
- **TN**: Healthy correctly cleared


In [ ]:
cm_data = np.array([[test_m['tn'], test_m['fp']],
                    [test_m['fn'], test_m['tp']]])
class_names = ['Healthy (0)', 'DR (1)']

fig, ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(cm_data, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(class_names); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title(f'Confusion Matrix  (thr={best_threshold:.2f})')
thresh_cm = cm_data.max()/2.0
for i in range(2):
    for j in range(2):
        ax.text(j,i,str(cm_data[i,j]),ha='center',va='center',
                color='white' if cm_data[i,j]>thresh_cm else 'black', fontsize=14)
# Annotations
labels_ann = [['TN','FP'],['FN','TP']]
for i in range(2):
    for j in range(2):
        ax.text(j,i+0.3,f'({labels_ann[i][j]})',ha='center',va='center',
                color='white' if cm_data[i,j]>thresh_cm else 'black', fontsize=9, alpha=0.8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'figures'/'confusion_matrix.png',dpi=150); plt.show()
print('Saved → outputs/figures/confusion_matrix.png')


## 18) Metrics Summary Table


In [ ]:
rows = [
    ('ROC AUC (threshold-independent)',  f"{test_m.get('roc_auc','—'):.4f}" if 'roc_auc' in test_m else '—'),
    ('PR  AUC (threshold-independent)',  f"{test_m.get('pr_auc','—'):.4f}" if 'pr_auc' in test_m else '—'),
    ('Decision threshold',               f"{test_m['threshold']:.3f}"),
    ('Recall (Sensitivity)',             f"{test_m['recall']:.4f}"),
    ('Specificity (1-FPR)',              f"{test_m['specificity']:.4f}"),
    ('Precision',                        f"{test_m['precision']:.4f}"),
    ('F1 Score',                         f"{test_m['f1']:.4f}"),
    ('Balanced Accuracy',                f"{test_m['balanced_accuracy']:.4f}"),
    ('True  Positives (TP)',             str(test_m['tp'])),
    ('True  Negatives (TN)',             str(test_m['tn'])),
    ('False Positives (FP)',             str(test_m['fp'])),
    ('False Negatives (FN)',             str(test_m['fn'])),
]

print(f"{'Metric':<40} {'Value':>10}")
print('-'*52)
for name, val in rows:
    print(f"{name:<40} {val:>10}")

# Save JSON
metrics_out = {k: (float(v) if isinstance(v,(float,int,np.floating,np.integer)) else v)
               for k,v in test_m.items()}
with open(OUTPUT_DIR/'tables'/'test_metrics.json','w') as f:
    json.dump(metrics_out, f, indent=2)
print('\nSaved → outputs/tables/test_metrics.json')


## 19) Error Case Analysis (FP / FN)

Exports a CSV listing every test sample with TP / TN / FP / FN label and predicted probability.


In [ ]:
error_rows = []
for i, s in enumerate(test_samples):
    true_lbl = int(test_labels_a[i])
    pred_lbl = int(test_preds_a[i])
    prob     = float(test_probs_a[i])
    if   true_lbl==1 and pred_lbl==1: outcome='TP'
    elif true_lbl==0 and pred_lbl==0: outcome='TN'
    elif true_lbl==0 and pred_lbl==1: outcome='FP'
    else:                              outcome='FN'
    error_rows.append({'sample_id':i,'patient_id':s.patient_id,
                       'true_label':true_lbl,'predicted_label':pred_lbl,
                       'probability':round(prob,4),'outcome':outcome})

csv_path = OUTPUT_DIR/'tables'/'error_cases.csv'
with open(csv_path,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=error_rows[0].keys())
    w.writeheader(); w.writerows(error_rows)

# Summary
for outcome in ['TP','TN','FP','FN']:
    cnt = sum(1 for r in error_rows if r['outcome']==outcome)
    avg_prob = np.mean([r['probability'] for r in error_rows if r['outcome']==outcome]) if cnt else 0
    print(f'  {outcome}: {cnt} cases  (avg prob={avg_prob:.3f})')
print(f'\nSaved → {csv_path}')


## 20) Save Model Checkpoint


In [ ]:
if TORCH_AVAILABLE and best_state:
    ckpt_path = OUTPUT_DIR/'checkpoints'/'best_model.pt'
    torch.save(best_state, ckpt_path)
    print(f'Model checkpoint saved → {ckpt_path}')
    # Also save hyperparameters alongside checkpoint
    hp = dict(IMG_SIZE=IMG_SIZE,BATCH_SIZE=BATCH_SIZE,LR=LR,EPOCHS=EPOCHS,
              PATIENCE=PATIENCE,UNFREEZE_EPOCH=UNFREEZE_EPOCH,
              AUG_MODE=AUG_MODE,LOSS_FN=LOSS_FN,SEED=SEED,
              best_epoch=best_epoch,best_val_bacc=float(best_val_bacc),
              threshold=float(best_threshold))
    with open(OUTPUT_DIR/'checkpoints'/'hparams.json','w') as f:
        json.dump(hp, f, indent=2)
    print(f'Hyperparameters saved → {OUTPUT_DIR}/checkpoints/hparams.json')
else:
    print('ℹ️  No trained model to save (DEMO_MODE or PyTorch not available).')


## 21) 5-Fold Cross-Validation (Patient-Level)

Provides a more robust estimate of generalisation performance on a small dataset.
Fold splits are made at the **patient level** to prevent leakage.


In [ ]:
def stratified_kfold_patients(samples, n_folds=5, seed=42):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for s in samples: by_label[s.label].append(s.patient_id)
    class_folds = {}
    for label, pids in by_label.items():
        unique = list(dict.fromkeys(pids)); rng.shuffle(unique)
        n = len(unique); fold_sz = n // n_folds
        folds = [unique[i*fold_sz:(i+1)*fold_sz] for i in range(n_folds)]
        for i, pid in enumerate(unique[n_folds*fold_sz:]): folds[i].append(pid)
        class_folds[label] = folds
    splits = []
    for fi in range(n_folds):
        val_ids  = set(pid for folds in class_folds.values() for pid in folds[fi])
        train_ids = set(pid for label,folds in class_folds.items()
                       for i,f in enumerate(folds) if i!=fi for pid in f)
        splits.append(([s for s in samples if s.patient_id in train_ids],
                       [s for s in samples if s.patient_id in val_ids]))
    return splits

fold_results = []

if TORCH_AVAILABLE:
    folds = stratified_kfold_patients(all_samples, CV_FOLDS, SEED)
    for fold_idx, (tr_s, va_s) in enumerate(folds):
        print(f'\n{"="*50}')
        print(f'Fold {fold_idx+1}/{CV_FOLDS}  train={len(tr_s)} val={len(va_s)}')
        _cc = [sum(1 for s in tr_s if s.label==c) for c in [0,1]]
        _sw = [1/_cc[s.label] for s in tr_s]
        _sam = WeightedRandomSampler(_sw, len(_sw), True)
        _tr_ds = OCTOCTADatasetV2(tr_s, policy=train_policy)
        _va_ds = OCTOCTADatasetV2(va_s, policy=None)
        _tr_ld = DataLoader(_tr_ds, BATCH_SIZE, sampler=_sam, num_workers=NUM_WORKERS)
        _va_ld = DataLoader(_va_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        _m = DualResNet(True).to(device)
        _pw = _cc[0]/max(_cc[1],1)
        _crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([_pw], device=device))
        _opt = optim.Adam(filter(lambda p:p.requires_grad,_m.parameters()), lr=LR)
        _best_bacc, _best_p, _best_l, _pat, _best_st = -1.0,[],[],0,None
        for ep in range(1, EPOCHS+1):
            if ep==UNFREEZE_EPOCH: _m.unfreeze_backbones(); _opt=optim.Adam(_m.parameters(),lr=LR*0.1)
            run_epoch(_m,_tr_ld,_crit,_opt,device,True)
            _,vl,vp = run_epoch(_m,_va_ld,_crit,None,device,False)
            vm2 = binary_metrics(vl,vp)
            if vm2['balanced_accuracy']>_best_bacc:
                _best_bacc=vm2['balanced_accuracy'];_best_p=list(vp);_best_l=list(vl)
                _best_st={k:v.cpu().clone() for k,v in _m.state_dict().items()};_pat=0
            else: _pat+=1
            if _pat>=PATIENCE: break
        thr_f = find_screening_threshold(_best_l,_best_p,MIN_RECALL)
        fm2 = binary_metrics(_best_l,_best_p,thr_f)
        if SKLEARN_AVAILABLE and len(set(_best_l))>1:
            fm2['roc_auc'] = float(roc_auc_score(_best_l,_best_p))
            fm2['pr_auc']  = float(average_precision_score(_best_l,_best_p))
        fm2['threshold']=thr_f; fm2['fold']=fold_idx+1
        fold_results.append(fm2)
        print(f'  Fold {fold_idx+1}: F1={fm2["f1"]:.3f} Recall={fm2["recall"]:.3f} '
              f'Spec={fm2["specificity"]:.3f} AUC={fm2.get("roc_auc",0):.3f}')
else:
    # DEMO fold results
    fold_results = [
        dict(fold=1,recall=0.917,specificity=0.923,precision=0.846,f1=0.880,balanced_accuracy=0.920,roc_auc=0.968,pr_auc=0.921,threshold=0.45),
        dict(fold=2,recall=0.833,specificity=0.956,precision=0.909,f1=0.870,balanced_accuracy=0.895,roc_auc=0.952,pr_auc=0.912,threshold=0.52),
        dict(fold=3,recall=0.944,specificity=0.934,precision=0.872,f1=0.906,balanced_accuracy=0.939,roc_auc=0.981,pr_auc=0.947,threshold=0.43),
        dict(fold=4,recall=0.900,specificity=0.945,precision=0.900,f1=0.900,balanced_accuracy=0.922,roc_auc=0.971,pr_auc=0.935,threshold=0.47),
        dict(fold=5,recall=0.889,specificity=0.912,precision=0.842,f1=0.865,balanced_accuracy=0.900,roc_auc=0.958,pr_auc=0.908,threshold=0.50),
    ]
    print('ℹ️  [DEMO] 5-fold cross-validation results loaded')


## 22) Cross-Validation Summary Table


In [ ]:
metric_keys = ['recall','specificity','f1','balanced_accuracy','roc_auc','pr_auc']
means = {k: np.mean([r[k] for r in fold_results if k in r]) for k in metric_keys}
stds  = {k: np.std( [r[k] for r in fold_results if k in r]) for k in metric_keys}

print('='*60)
print(f' {CV_FOLDS}-FOLD CROSS-VALIDATION SUMMARY')
print('='*60)
print(f"{'Metric':<25} {'Mean':>8} {'± Std':>8}")
print('-'*45)
for k in metric_keys:
    print(f'  {k:<23} {means[k]:>8.4f} ± {stds[k]:.4f}')
print()
print(f"{'Fold':<6} {'Recall':>8} {'Spec':>8} {'F1':>8} {'BAcc':>8} {'AUC':>8}")
print('-'*50)
for r in fold_results:
    print(f"  {r['fold']:<4} {r['recall']:>8.3f} {r['specificity']:>8.3f} "
          f"{r['f1']:>8.3f} {r['balanced_accuracy']:>8.3f} {r.get('roc_auc',0):>8.3f}")

# Save CSV
cv_csv = OUTPUT_DIR/'tables'/'cv_fold_results.csv'
with open(cv_csv,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=fold_results[0].keys())
    w.writeheader(); w.writerows(fold_results)
print(f'\nSaved → {cv_csv}')


## 23) Cross-Validation Bar Chart


In [ ]:
x = np.arange(len(metric_keys))
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x, [means[k] for k in metric_keys], yerr=[stds[k] for k in metric_keys],
       capsize=5, color='steelblue', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(metric_keys, rotation=25, ha='right')
ax.set_ylabel('Score'); ax.set_title(f'{CV_FOLDS}-Fold CV Summary (mean ± std)')
ax.set_ylim([0,1.15])
for i,(k,m,s) in enumerate(zip(metric_keys,[means[k] for k in metric_keys],[stds[k] for k in metric_keys])):
    ax.text(i, m+s+0.02, f'{m:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'figures'/'cv_summary_barplot.png',dpi=150); plt.show()
print('Saved → outputs/figures/cv_summary_barplot.png')


## 24) Augmentation Ablation Study

Compares **none / basic_all / basic_dr_only** on the same train/val/test split
and the same random seed for a valid comparison.


In [ ]:
ablation_modes = ['none', 'basic_all', 'basic_dr_only']
ablation_results = {}

if TORCH_AVAILABLE:
    for mode in ablation_modes:
        print(f'\nAblation: mode={mode!r}')
        _pol = get_augmentation_policy(mode) if AUG_AVAILABLE else None
        _tr_ds = OCTOCTADatasetV2(train_samples, policy=_pol)
        _va_ds = OCTOCTADatasetV2(val_samples,   policy=None)
        _te_ds = OCTOCTADatasetV2(test_samples,  policy=None)
        _tr_ld = DataLoader(_tr_ds, BATCH_SIZE, sampler=WeightedRandomSampler(sw, len(sw), True),
                            num_workers=NUM_WORKERS)
        _va_ld = DataLoader(_va_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        _te_ld = DataLoader(_te_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        _m  = DualResNet(True).to(device)
        _crit = nn.BCEWithLogitsLoss(pos_weight=pw_tensor)
        _opt  = optim.Adam(filter(lambda p:p.requires_grad,_m.parameters()), lr=LR)
        _best_bacc, _best_p, _best_l, _pat, _best_st = -1.0,[],[],0,None
        for ep in range(1, EPOCHS+1):
            if ep==UNFREEZE_EPOCH: _m.unfreeze_backbones(); _opt=optim.Adam(_m.parameters(),lr=LR*0.1)
            run_epoch(_m,_tr_ld,_crit,_opt,device,True)
            _,vl,vp = run_epoch(_m,_va_ld,_crit,None,device,False)
            vm2 = binary_metrics(vl,vp)
            if vm2['balanced_accuracy']>_best_bacc:
                _best_bacc=vm2['balanced_accuracy'];_best_p=list(vp);_best_l=list(vl)
                _best_st={k:v.cpu().clone() for k,v in _m.state_dict().items()};_pat=0
            else: _pat+=1
            if _pat>=PATIENCE: break
        _m.load_state_dict(_best_st)
        thr_a = find_screening_threshold(_best_l,_best_p,MIN_RECALL)
        _,te_lbl,te_prb = run_epoch(_m,_te_ld,_crit,None,device,False)
        am = binary_metrics(te_lbl,te_prb,thr_a)
        if SKLEARN_AVAILABLE and len(set(te_lbl))>1:
            am['roc_auc']=float(roc_auc_score(te_lbl,te_prb))
            am['pr_auc'] =float(average_precision_score(te_lbl,te_prb))
        ablation_results[mode]=am
        print(f'  recall={am["recall"]:.3f} spec={am["specificity"]:.3f} '
              f'f1={am["f1"]:.3f} auc={am.get("roc_auc",0):.3f}')
else:
    # DEMO ablation results
    ablation_results = {
        'none':         dict(recall=0.800,specificity=0.934,f1=0.842,balanced_accuracy=0.867,roc_auc=0.934,pr_auc=0.891),
        'basic_all':    dict(recall=0.943,specificity=0.934,f1=0.892,balanced_accuracy=0.938,roc_auc=0.979,pr_auc=0.939),
        'basic_dr_only':dict(recall=0.914,specificity=0.956,f1=0.901,balanced_accuracy=0.935,roc_auc=0.972,pr_auc=0.928),
    }
    print('ℹ️  [DEMO] Augmentation ablation results loaded')


## 25) Augmentation Ablation Comparison Chart


In [ ]:
abl_metric_keys = ['recall','specificity','f1','balanced_accuracy','roc_auc','pr_auc']
abl_colors = ['steelblue','darkorange','forestgreen']
x = np.arange(len(abl_metric_keys)); w = 0.25

print(f"{'Mode':<20} "+" ".join(f"{k:>10}" for k in abl_metric_keys))
print('-'*90)
for mode, m in ablation_results.items():
    vals = [m.get(k,0) for k in abl_metric_keys]
    print(f"{mode:<20} "+" ".join(f"{v:>10.4f}" for v in vals))

fig, ax = plt.subplots(figsize=(12,5))
for i,(mode,m) in enumerate(ablation_results.items()):
    vals = [m.get(k,0) for k in abl_metric_keys]
    ax.bar(x+i*w, vals, w, label=mode, color=abl_colors[i], alpha=0.85)
ax.set_xticks(x+w); ax.set_xticklabels(abl_metric_keys, rotation=20, ha='right')
ax.set_ylabel('Score'); ax.set_title('Augmentation Ablation Study')
ax.set_ylim([0,1.15]); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'figures'/'augmentation_ablation_comparison.png',dpi=150); plt.show()
print('Saved → outputs/figures/augmentation_ablation_comparison.png')

# Save CSV
abl_rows = [{'augmentation_mode':k,**v} for k,v in ablation_results.items()]
with open(OUTPUT_DIR/'tables'/'augmentation_ablation_results.csv','w',newline='') as f:
    w2 = csv.DictWriter(f, fieldnames=abl_rows[0].keys())
    w2.writeheader(); w2.writerows(abl_rows)
print('Saved → outputs/tables/augmentation_ablation_results.csv')


## 26) Before vs After Cross-Validation Comparison

Augmentation on same split, same seed for valid comparison.


In [ ]:
# Augmentation ablation on the same train/val/test split
# Shows how each mode compares fold-by-fold on the key screening metric (recall).

# For the before/after comparison we show the single train/val/test split
# with no-augmentation baseline vs best augmentation mode.
modes_cmp = list(ablation_results.keys())
recall_vals   = [ablation_results[m]['recall']           for m in modes_cmp]
spec_vals     = [ablation_results[m]['specificity']      for m in modes_cmp]
bacc_vals     = [ablation_results[m]['balanced_accuracy'] for m in modes_cmp]
auc_vals      = [ablation_results[m].get('roc_auc',0)    for m in modes_cmp]

x_cmp = np.arange(len(modes_cmp))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Recall and Specificity
axes[0].bar(x_cmp-0.2, recall_vals, 0.35, label='Recall',      color='tomato',    alpha=0.85)
axes[0].bar(x_cmp+0.2, spec_vals,   0.35, label='Specificity', color='steelblue', alpha=0.85)
axes[0].axhline(MIN_RECALL, color='red', linestyle='--', lw=1, label=f'Min recall={MIN_RECALL}')
axes[0].set_xticks(x_cmp); axes[0].set_xticklabels(modes_cmp, rotation=10)
axes[0].set_ylim([0,1.15]); axes[0].set_title('Recall vs Specificity')
axes[0].legend()
for i in x_cmp:
    axes[0].text(i-0.2, recall_vals[i]+0.02, f'{recall_vals[i]:.3f}', ha='center', fontsize=9)
    axes[0].text(i+0.2, spec_vals[i]+0.02,   f'{spec_vals[i]:.3f}',   ha='center', fontsize=9)

# Right: Balanced Accuracy and ROC AUC
axes[1].bar(x_cmp-0.2, bacc_vals, 0.35, label='Balanced Acc', color='darkorange', alpha=0.85)
axes[1].bar(x_cmp+0.2, auc_vals,  0.35, label='ROC AUC',      color='purple',     alpha=0.85)
axes[1].set_xticks(x_cmp); axes[1].set_xticklabels(modes_cmp, rotation=10)
axes[1].set_ylim([0,1.15]); axes[1].set_title('Balanced Accuracy vs ROC AUC')
axes[1].legend()
for i in x_cmp:
    axes[1].text(i-0.2, bacc_vals[i]+0.02, f'{bacc_vals[i]:.3f}', ha='center', fontsize=9)
    axes[1].text(i+0.2, auc_vals[i]+0.02,  f'{auc_vals[i]:.3f}',  ha='center', fontsize=9)

plt.suptitle('Before vs After Augmentation (same split, same seed)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'figures'/'before_after_augmentation.png',dpi=150); plt.show()
print('Saved → outputs/figures/before_after_augmentation.png')


## 27) All Outputs Summary

Lists every file that was written during this run.


In [ ]:
print('='*55)
print(' GENERATED OUTPUTS')
print('='*55)
for sub in ['figures','tables','reports','checkpoints']:
    sub_path = OUTPUT_DIR / sub
    files = sorted(sub_path.glob('*')) if sub_path.exists() else []
    if files:
        print(f'\n{sub}/')
        for f in files: print(f'  {f.name}')

print('\n' + '='*55)
print(' NEXT STEPS')
print('='*55)
print("  1. Review outputs/tables/test_metrics.json for your thesis numbers")
print("  2. Include figures/roc_curve.png + figures/confusion_matrix.png in your report")
print("  3. Reference figures/augmentation_ablation_comparison.png for ablation section")
print("  4. outputs/checkpoints/best_model.pt is your saved model weights")
print("  5. Run with DEMO_MODE=False and your real Final_Dataset to get true results")
